# Labelling Strategy

This notebook implements a **weak-supervision labelling strategy** for FIRMS (Fire Information for Resource Management System) hotspot data over India.

The core idea:
- Group satellite detections by approximate location (`group_id` = lat/lon rounded to 3 decimal places)
- Compute per-group persistence and seasonality statistics
- Cross-reference with OpenStreetMap (OSM) industrial/mining features to assign probabilistic labels

## Outputs
- `thermalwatch_labeled.parquet` — labeled training set (used by next notebook)
- `label_summary.json` — labeling audit metadata

## Labels
| Label | Description |
|---|---|
| `industrial_thermal_source` | Persistent, monsoon-active, near industrial OSM feature |
| `mining_thermal_source` | Same as above, but nearest OSM type is `landuse_quarry` |
| `natural_fire` | Seasonal (≤3 months active), not near industry |
| `unknown` | Everything else |

> **Data assumption:** The raw FIRMS parquet is located at `data/processed/firms_india` relative to the repo root. Adjust `DATA_PATH` below if needed.

## 0. Configuration

Set paths here. All paths are relative to the repo root by default.

In [ ]:
import os

# Path to the processed FIRMS parquet directory (partitioned)
DATA_PATH = os.environ.get("FIRMS_INDIA_PATH", "../data/processed/firms_india")

# Path to the OSM industrial features JSON exported from Overpass API
OSM_JSON_PATH = os.environ.get("OSM_JSON_PATH", "../data/raw/osm_industrial_india_raw.json")

# Output
OUTPUT_PARQUET = "thermalwatch_labeled.parquet"
OUTPUT_SUMMARY = "label_summary.json"

print(f"FIRMS data path : {DATA_PATH}")
print(f"OSM JSON path   : {OSM_JSON_PATH}")

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import pyarrow.parquet as pq
import pyarrow.dataset as ds
import json

## 2. Load FIRMS Data

Read the partitioned parquet dataset of FIRMS India hotspot detections.

In [ ]:
df = pd.read_parquet(DATA_PATH, engine="pyarrow")
df.shape

## 3. Spatial Grouping

Create a `group_id` by rounding lat/lon to 3 decimal places (~100 m resolution).
Each unique `group_id` represents a distinct physical location that has been detected multiple times.

In [ ]:
df['group_id'] = df['latitude'].round(3).astype(str) + '_' + df['longitude'].round(3).astype(str)
df['group_id'].nunique()

## 4. Group-Level Statistics

Aggregate per-group statistics: observation count, first/last detection, mean/std FRP (Fire Radiative Power).

In [ ]:
group_stats = df.groupby('group_id').agg(
    obs_count=('observed_at', 'count'),
    first_seen=('observed_at', 'min'),
    last_seen=('observed_at', 'max'),
    mean_frp=('frp', 'mean'),
    std_frp=('frp', 'std'),
    latitude=('latitude', 'first'),
    longitude=('longitude', 'first')
).reset_index()

group_stats.shape
group_stats.sort_values('obs_count', ascending=False).head(10)

## 5. Persistence & Seasonality Features

Compute:
- `months_active`: number of distinct calendar months with detections
- `monsoon_active`: whether the group was detected during monsoon (Jun–Sep)
- `frp_cv`: coefficient of variation of FRP (indicates variability)

In [ ]:
df['month'] = df['observed_at'].dt.month
df['is_monsoon'] = df['month'].between(6, 9)

persistence = df.groupby('group_id').agg(
    months_active=('month', lambda x: x.nunique()),
    monsoon_obs_count=('is_monsoon', 'sum'),
).reset_index()

group_stats = group_stats.merge(persistence, on='group_id')

group_stats['monsoon_active'] = group_stats['monsoon_obs_count'] > 0
group_stats['frp_cv'] = group_stats['std_frp'] / group_stats['mean_frp']

group_stats.sort_values('obs_count', ascending=False).head(10)[
    ['group_id', 'obs_count', 'months_active', 'monsoon_active', 'frp_cv']
]

In [ ]:
group_stats['months_active'].value_counts().sort_index()

In [ ]:
group_stats['monsoon_active'].value_counts()

In [ ]:
group_stats['obs_count'].describe()

### Persistent groups
How many groups are active ≥9 months AND monsoon-active? These are strong industrial candidates.

In [ ]:
((group_stats['months_active'] >= 9) & (group_stats['monsoon_active'])).sum()

## 6. OSM Industrial Feature Cross-Reference

Load the OpenStreetMap industrial features (exported via Overpass API) and build a KD-tree for nearest-neighbor lookup.

**OSM tags used:**
- `power=*` (power plants, substations)
- `landuse=industrial` / `landuse=quarry`
- `man_made=chimney` / `man_made=works`
- `industrial=*`

In [ ]:
with open(OSM_JSON_PATH) as f:
    osm_raw = json.load(f)

len(osm_raw['elements'])

In [ ]:
osm_features = []

for el in osm_raw['elements']:
    tags = el.get('tags', {})
    
    if el['type'] == 'node':
        lat, lon = el.get('lat'), el.get('lon')
    else:  # way — use center
        center = el.get('center', {})
        lat, lon = center.get('lat'), center.get('lon')
    
    if lat is None or lon is None:
        continue
    
    # determine primary tag type
    if 'power' in tags:
        feature_type = f"power_{tags['power']}"
    elif 'landuse' in tags:
        feature_type = f"landuse_{tags['landuse']}"
    elif 'man_made' in tags:
        feature_type = f"man_made_{tags['man_made']}"
    elif 'industrial' in tags:
        feature_type = f"industrial_{tags['industrial']}"
    else:
        feature_type = "other"
    
    osm_features.append({
        'osm_id': el['id'],
        'lat': lat,
        'lon': lon,
        'feature_type': feature_type,
        'name': tags.get('name', 'Unnamed')
    })

osm_df = pd.DataFrame(osm_features)
osm_df.shape
osm_df['feature_type'].value_counts()

### KD-Tree nearest-neighbor lookup

For each FIRMS group, find the nearest OSM industrial feature (by Euclidean distance in degrees).

In [ ]:
from scipy.spatial import cKDTree

# Build KD-tree from OSM feature coordinates
osm_coords = osm_df[['lat', 'lon']].values
tree = cKDTree(osm_coords)

# Query nearest OSM feature for every group
group_coords = group_stats[['latitude', 'longitude']].values
distances, indices = tree.query(group_coords, k=1)

group_stats['nearest_osm_distance_deg'] = distances
group_stats['nearest_osm_type'] = osm_df.iloc[indices]['feature_type'].values
group_stats['nearest_osm_name'] = osm_df.iloc[indices]['name'].values

group_stats[['group_id', 'nearest_osm_distance_deg', 'nearest_osm_type']].head(10)

In [ ]:
group_stats['nearest_osm_distance_km'] = group_stats['nearest_osm_distance_deg'] * 111
group_stats['near_industrial'] = group_stats['nearest_osm_distance_km'] <= 2.0  # 2km threshold

group_stats['near_industrial'].value_counts()

### OSM corroboration rate

Of the persistent (≥9 months active + monsoon active) groups, what fraction are near industrial OSM features?
A high rate validates the labeling approach.

In [ ]:
persistent_mask = (group_stats['months_active'] >= 9) & (group_stats['monsoon_active'])
group_stats[persistent_mask]['near_industrial'].value_counts()

In [ ]:
group_stats[persistent_mask]['near_industrial'].value_counts(normalize=True)

## 7. Label Assignment

Apply rule-based weak-supervision labels using three conditions:

| Priority | Condition | Label | Confidence |
|---|---|---|---|
| 1 | `months_active ≥ 9` AND `monsoon_active` AND `near_industrial` | `industrial_thermal_source` | 0.90 |
| 2 | `months_active ≥ 9` AND `monsoon_active` AND NOT `near_industrial` | `unknown` | 0.40 |
| 3 | `months_active ≤ 3` AND NOT `near_industrial` | `seasonal_fire_candidate` | 0.50 |
| default | everything else | `unknown` | 0.30 |

In [ ]:
is_persistent = (group_stats['months_active'] >= 9) & (group_stats['monsoon_active'])
is_seasonal = group_stats['months_active'] <= 3
near_industrial = group_stats['near_industrial']

conditions = [
    is_persistent & near_industrial,
    is_persistent & ~near_industrial,
    is_seasonal & ~near_industrial,
]

labels = ['industrial_thermal_source', 'unknown', 'seasonal_fire_candidate']
confidences = [0.9, 0.4, 0.5]
reasons = ['persistent_and_near_industrial', 'persistent_but_no_industrial_corroboration', 'seasonal_no_industrial_context']

group_stats['label'] = np.select(conditions, labels, default='unknown')
group_stats['label_confidence'] = np.select(conditions, confidences, default=0.3)
group_stats['label_reason'] = np.select(conditions, reasons, default='ambiguous_pattern')

group_stats['label'].value_counts()

### Refine: split industrial → mining

If the nearest OSM feature type is `landuse_quarry`, re-label as `mining_thermal_source`.

In [ ]:
def refine_industrial_label(row):
    if row['label'] == 'industrial_thermal_source':
        if row['nearest_osm_type'] == 'landuse_quarry':
            return 'mining_thermal_source'
        else:
            return 'industrial_thermal_source'
    return row['label']

group_stats['label'] = group_stats.apply(refine_industrial_label, axis=1)
group_stats['label'].value_counts()

### Rename seasonal candidate → natural_fire

In [ ]:
group_stats['label'] = group_stats['label'].replace('seasonal_fire_candidate', 'natural_fire')
group_stats['label'].value_counts()

### Label quality check: obs_count distributions per class

In [ ]:
group_stats.groupby('label')['obs_count'].describe()

## 8. Training Set Construction

Keep all industrial + mining samples (rare minority classes). 
Downsample `natural_fire` **stratified by first-detection month** to preserve seasonal distribution.
Downsample `unknown` to a smaller budget (least reliable class).

In [ ]:
natural_all = group_stats[group_stats['label'] == 'natural_fire']
natural_sample = natural_all.groupby(natural_all['first_seen'].dt.month, group_keys=False).apply(
    lambda x: x.sample(min(len(x), 2000), random_state=42)
)
natural_sample['label'].value_counts()
len(natural_sample)

In [ ]:
industrial = group_stats[group_stats['label'] == 'industrial_thermal_source']
mining = group_stats[group_stats['label'] == 'mining_thermal_source']

natural_all = group_stats[group_stats['label'] == 'natural_fire']
natural_sample = natural_all.groupby(natural_all['first_seen'].dt.month, group_keys=False).apply(
    lambda x: x.sample(min(len(x), 2000), random_state=42)
)

unknown_sample = group_stats[group_stats['label'] == 'unknown'].sample(n=10000, random_state=42)

training_set = pd.concat([industrial, mining, natural_sample, unknown_sample], ignore_index=True)
training_set['label'].value_counts()

## 9. Save Outputs

In [ ]:
training_set.to_parquet(OUTPUT_PARQUET, engine="pyarrow", index=False)
print(f"Saved {len(training_set)} rows to {OUTPUT_PARQUET}")

In [ ]:
label_summary = {
    "total_groups_in_full_dataset": int(len(group_stats)),
    "total_training_groups": int(len(training_set)),
    "class_distribution": training_set['label'].value_counts().to_dict(),
    "osm_corroboration_rate": float((group_stats[persistent_mask]['near_industrial']).mean()),
    "labeling_method": "weak_supervision: FIRMS persistence + OSM industrial proximity",
    "osm_features_used": int(len(osm_df)),
    "notes": "natural_fire stratified by first-detection month to preserve seasonal distribution; industrial/mining classes use full available population due to small size"
}

with open(OUTPUT_SUMMARY, "w") as f:
    json.dump(label_summary, f, indent=2)

label_summary